In [ ]:
# import the necessary libraries
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
# load the data
data = pd.read_csv('/content/companies.csv')

#1.Explore the data set

In [ ]:
data.info() # View data types and non-null counts of each column.

In [ ]:
data.head() # Display the first few rows of the dataset.

In [ ]:
data.describe() # Get summary statistics for numerical columns (mean, median, quartiles, etc.).

###Identify missing values

In [ ]:
data.isnull().sum() #identify missing values.

In [ ]:
import matplotlib.pyplot as plt
# Assuming 'data' is your DataFrame
plt.figure(figsize=(12, 8))  # Adjust the figure size as needed
sns.heatmap(data.isnull(), cbar=False, cmap='viridis')

# sns.heatmap(...): This creates the heatmap.
'''data.isnull() means This generates a DataFrame of the same shape as data,
 with True where the data is missing and False where it's not.'''
# cbar=False means Removes the color bar, which is not very useful in this context.
# cmap='viridis' means Sets the color map. where we can choose different color maps like 'YlGnBu', 'coolwarm', etc.

plt.title('Missing Data Heatmap')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.show()

###Understand the Distribution of Variables

In [ ]:
# view the distribution of numerical variables
data.hist(figsize=(14, 12))

In [ ]:
print(data.columns)

In [ ]:
data['twitter_username'].value_counts() # see the frequency of each category.

###Explore Relationships Between Variables

In [ ]:
data.dtypes

In [ ]:
'''calculate the correlation between numerical variables'''
# Filter out only numeric columns
numeric_data = data.select_dtypes(include=['number'])

# Calculate the correlation matrix
correlation_matrix = numeric_data.corr()

# Display the correlation matrix
correlation_matrix


In [ ]:
# visualize correlation_matrix with a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')

plt.title('Correlation Matrix Heatmap')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# visualize relationships between multiple variables simultaneously.
sns.pairplot(data)

plt.show()


In [ ]:
sns.scatterplot(data) # Visualizing relationships between two numerical variables.
plt.show()

###Here check for Data Imbalances in data processing

In [ ]:
data['twitter_username'].value_counts()

In [ ]:
# Group Analysis: Compare summary statistics across different groups
numeric_data = data.select_dtypes(include=['number'])

# Assuming 'twitter_username' is the column you want to group by
mean_values = numeric_data.groupby(data['twitter_username']).mean()
mean_values

###Outlier Detection:
####for a Single Column

In [ ]:
# Assuming 'twitter_username' is the column i want to visualize
plt.figure(figsize=(10, 6))
sns.boxplot(x=data['twitter_username'])

plt.title('Boxplot of column_name')
plt.xlabel('column_name')
plt.show()


####Example for Multiple Columns:

In [ ]:
# Creating boxplots for all numeric columns
plt.figure(figsize=(12, 8))
sns.boxplot(data=data.select_dtypes(include=['number']))

plt.title('Boxplots of Numeric Columns')
plt.show()


In [ ]:
data.columns

####Example with Grouping (Categorical and Numeric):

In [ ]:
# Assuming 'relationships' is the categorical column and 'value' is the numeric column
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
sns.boxplot(x='twitter_username', y='city', data=data)

plt.title('Boxplot of value by twitter_data')
plt.xlabel('twitter_username')
plt.ylabel('city')
plt.show()

In [ ]:
data.isnull().sum() #identify missing values.



*   Remove Rows/Columns with Missing Data



In [ ]:
data.dropna(axis=0, inplace=True)  # Drop rows with missing values
data.dropna(axis=1, inplace=True)  # Drop columns with missing values




*   Impute Missing Values:
    *   Fill missing values with mean, median, mode, or a specific value.







In [ ]:
data['twitter_username'].fillna(data['twitter_username'].mean(), inplace=True)
data['twitter_username'].fillna(data['twitter_username'].median(), inplace=True)
data['twitter_username'].fillna('Unknown', inplace=True)


In [ ]:
# Replace incorrect values manually or using regex.
data['twitter_username'] = data['twitter_username'].replace('errored_value', 'correct_value')


In [ ]:
# Identify and remove duplicate rows.
data.drop_duplicates(inplace=True)


In [ ]:
# Outlier handling: z-score filtering only makes sense on genuinely
# numeric columns (e.g. funding amounts) - NOT on an encoded username.
from scipy import stats

numeric_cols = data.select_dtypes(include=['number']).columns
if len(numeric_cols) > 0:
    z = np.abs(stats.zscore(data[numeric_cols], nan_policy='omit'))
    data = data[(z < 3).all(axis=1)]
print('rows after outlier removal:', len(data))


In [ ]:
import numpy as np

# Calculate Q1 (25th percentile) and Q3 (75th percentile)
Q1 = data['twitter_username'].quantile(0.25)
Q3 = data['twitter_username'].quantile(0.75)

# Calculate the Interquartile Range (IQR)
IQR = Q3 - Q1

# Define the upper and lower limits for capping
upper_limit = Q3 + 1.5 * IQR
lower_limit = Q1 - 1.5 * IQR

# Capping the outliers
data['twitter_username'] = np.where(data['twitter_username'] > upper_limit, upper_limit, data['twitter_username'])
data['twitter_username'] = np.where(data['twitter_username'] < lower_limit, lower_limit, data['twitter_username'])

# Check the result
print(data['twitter_username'].describe())



In [ ]:
# Drop columns that are irrelevant or redundant for the analysis.
data.drop(['region'], axis=1, inplace=True)


In [ ]:
data['twitter_username'] = data['twitter_username'].astype('float')
data['city'] = data['city'].astype('category')


In [ ]:
def clean_column(column):
    # Convert all entries to string type
    column = column.astype(str)

    # Remove leading/trailing whitespace
    column = column.str.strip()

    # Convert to lowercase
    column = column.str.lower()

    return column

# Apply the function to the 'twitter_username' column
data['cleaned_column'] = clean_column(data['twitter_username'])

# Check the result
print(data['cleaned_column'].head())


In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
company_names = data['name'].unique()
print(company_names)

In [ ]:
data.head()

# 3. Model Training & Evaluation

Everything above is exploration and cleaning. This section does the actual
prediction task: **classify each startup's `status`** (operating / acquired /
closed / ipo) from its numeric and categorical features.

The encoding, imputation, and scaling live *inside* an sklearn `Pipeline`
(see [`src/startup_pipeline.py`](src/startup_pipeline.py)), so they are fit on
the training split only - no leakage. Two models are compared, both with
balanced class weights because the status classes are imbalanced.


In [ ]:
import sys
sys.path.append('.')  # so `src` is importable when running the notebook
from src.startup_pipeline import load_data, train_and_evaluate

# Use the real target column from the Crunchbase export.
X, y = load_data('companies.csv', target='status')
print(f'rows: {len(X):,}')
print('class balance:')
print(y.value_counts())


In [ ]:
results, (numeric, categorical) = train_and_evaluate(X, y)
print('numeric features:', numeric)
print('categorical features:', categorical)


### Results

`train_and_evaluate` prints accuracy, macro-F1, a per-class classification
report, and a confusion matrix for each model. Paste the numbers into the
README's Results table. Macro-F1 is the honest headline here because the
classes are imbalanced - plain accuracy is inflated by the dominant
'operating' class.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

best = max(results, key=lambda k: results[k]['macro_f1'])
print(f'Best model by macro-F1: {best}')
cm = results[best]['confusion_matrix']
ConfusionMatrixDisplay(cm, display_labels=results[best]['labels']).plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {best}')
plt.tight_layout(); plt.show()
